# 聚合 real-video VAE latent probe shards

该 Notebook 只负责把 `shard_runs` 中的阶段二分片结果聚合为可供阶段三使用的正式结果包。它不执行 VAE runner。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动时使用的仓库 clone URL
REPO_DIR = "/content/TSTW-VW"  # Colab 本地仓库根目录
GIT_REF = "explicit-sync"  # 要拉取的项目分支
DRIVE_MOUNT = "/content/drive"  # Google Drive 挂载点
DRIVE_ROOT = "/content/drive/MyDrive"  # Google Drive MyDrive 根目录
LOCAL_RUNTIME_ROOT = "/content/TSTW_runtime"  # 本次 Colab session 的本地运行根目录
AUTO_RESOLVE_SHARD_ROOTS = True  # 是否自动搜索 shard_runs 下最新且完整的一组 shard。推荐保持 True。
SHARD_RUNS_ROOT = f"{DRIVE_ROOT}/TSTW/results/real_video_vae_latent_probe/shard_runs"  # 自动搜索 shard run family 目录的父目录
SHARD_ROOTS = []  # 可选手动覆盖。留空且 AUTO_RESOLVE_SHARD_ROOTS=True 时自动搜索; 非空时按该列表聚合。
REQUIRED_SHARD_SHORT_COMMIT = ""  # 可选过滤 short commit, 例如 "8e70039"; 留空表示自动选择最新完整组。
REQUIRED_SHARD_COUNT = None  # 可选过滤 shard_count, 例如 2; None 表示不限制。
MERGED_RUN_ROOT = f"{LOCAL_RUNTIME_ROOT}/runs/real_video_vae_latent_probe_shard_aggregated"  # 聚合后的本地 run_root
LOCAL_FAMILY_ROOT = f"{LOCAL_RUNTIME_ROOT}/families/real_video_vae_latent_probe/shard_aggregated"  # 聚合包先写入的本地 family 父目录
PROTOCOL_CONFIG = "configs/protocol/real_video_vae_latent_probe.json"  # 阶段二 fixed-FPR 协议配置
MECHANISM_CONFIG = "configs/protocol/stage2_mechanism_gate.json"  # 阶段二 mechanism gate 配置
RUNTIME_PROFILE = "formal"  # 聚合结果登记使用的 runtime profile
EXCLUDE_LARGE_INTERMEDIATE_LATENTS = True  # 聚合包默认瘦身, 不重复打包大型 latent/video 中间目录
OVERWRITE_DRIVE_RESULT = True  # 若 Drive 聚合目录已存在, 是否覆盖


In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed



def parse_json_process_stdout(stdout_text):
    """解析仓库脚本输出的 JSON。

    脚本通常使用 indent=2 打印多行 JSON, 因此不能只读取最后一行。
    该函数从完整 stdout 中定位第一个 JSON 对象并解析, 便于 Colab cell 稳定接收结果。
    """
    resolved_text = str(stdout_text).strip()
    if not resolved_text:
        raise ValueError("脚本 stdout 为空, 无法解析 JSON 结果。")
    json_start = resolved_text.find("{")
    if json_start < 0:
        raise ValueError("脚本 stdout 中未找到 JSON 对象。")
    return json.loads(resolved_text[json_start:])

def split_paths(raw_paths):
    if isinstance(raw_paths, (list, tuple)):
        return [str(item).strip() for item in raw_paths if str(item).strip()]
    return [item.strip() for item in re.split(r"[;,]", str(raw_paths)) if item.strip()]


def load_json_file(path):
    """读取 JSON 文件, 失败时返回空字典。"""
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return {}


def inspect_shard_family_root(family_root):
    """检查单个 shard family 目录是否可作为聚合输入。"""
    root = Path(family_root)
    checks = load_json_file(root / "family_checks.json")
    package_dir = root / "packages"
    zip_paths = sorted(package_dir.glob("*.zip"), key=lambda path: path.stat().st_mtime, reverse=True) if package_dir.exists() else []
    required_paths = checks.get("required_paths", {}) if isinstance(checks, dict) else {}
    status_value = checks.get("status") if isinstance(checks, dict) else None
    shard_count = checks.get("shard_count") if isinstance(checks, dict) else None
    shard_index = checks.get("shard_index") if isinstance(checks, dict) else None
    short_commit = checks.get("short_commit") if isinstance(checks, dict) else None
    if shard_count is None or shard_index is None or short_commit is None:
        match = re.search(r"_sc(?P<count>\d+)_si(?P<index>\d+)_(?P<commit>[0-9a-fA-F]+)$", root.name)
        if match:
            shard_count = shard_count if shard_count is not None else int(match.group("count"))
            shard_index = shard_index if shard_index is not None else int(match.group("index"))
            short_commit = short_commit or match.group("commit")
    return {
        "family_root": root,
        "name": root.name,
        "status": status_value,
        "shard_count": int(shard_count) if shard_count is not None else None,
        "shard_index": int(shard_index) if shard_index is not None else None,
        "short_commit": str(short_commit)[:7] if short_commit else None,
        "required_paths_ready": bool(required_paths) and all(bool(value) for value in required_paths.values()),
        "zip_ready": bool(zip_paths),
        "latest_mtime": max([root.stat().st_mtime] + [path.stat().st_mtime for path in zip_paths]),
    }


def resolve_latest_complete_shard_roots(shard_runs_root, required_short_commit="", required_shard_count=None):
    """自动搜索最新且完整的一组 shard family 目录。"""
    root = Path(shard_runs_root)
    if not root.exists():
        raise FileNotFoundError(root)
    shard_infos = [
        inspect_shard_family_root(path)
        for path in sorted(root.iterdir())
        if path.is_dir()
    ]
    usable_infos = [
        info for info in shard_infos
        if info["status"] == "shard_run_completed"
        and info["required_paths_ready"]
        and info["zip_ready"]
        and info["shard_count"] is not None
        and info["shard_index"] is not None
        and info["short_commit"]
    ]
    if required_short_commit:
        usable_infos = [info for info in usable_infos if info["short_commit"] == str(required_short_commit)[:7]]
    if required_shard_count is not None:
        usable_infos = [info for info in usable_infos if info["shard_count"] == int(required_shard_count)]
    grouped = {}
    for info in usable_infos:
        grouped.setdefault((info["short_commit"], info["shard_count"]), []).append(info)
    complete_groups = []
    for (commit, shard_count), infos in grouped.items():
        by_index = {info["shard_index"]: info for info in infos}
        expected_indexes = set(range(int(shard_count)))
        if set(by_index) != expected_indexes:
            continue
        complete_groups.append({
            "short_commit": commit,
            "shard_count": int(shard_count),
            "shard_indexes": sorted(by_index),
            "family_roots": [by_index[index]["family_root"] for index in sorted(by_index)],
            "latest_mtime": max(info["latest_mtime"] for info in by_index.values()),
        })
    if not complete_groups:
        raise FileNotFoundError({
            "reason": "no_complete_shard_group_found",
            "shard_runs_root": str(root),
            "required_short_commit": required_short_commit,
            "required_shard_count": required_shard_count,
            "candidate_count": len(shard_infos),
            "usable_count": len(usable_infos),
        })
    selected_group = sorted(complete_groups, key=lambda item: item["latest_mtime"], reverse=True)[0]
    return [path.as_posix() for path in selected_group["family_roots"]], selected_group


## 1. 挂载 Drive 并获取仓库

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")

repo_path = Path(REPO_DIR)
if repo_path.exists():
    run_command(["git", "fetch", "--depth", "1", "origin", GIT_REF], cwd=repo_path)
    run_command(["git", "checkout", GIT_REF], cwd=repo_path)
    run_command(["git", "pull", "--ff-only", "origin", GIT_REF], cwd=repo_path)
else:
    run_command(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, REPO_DIR])
short_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print({"short_commit": short_commit})


## 2. 聚合 shard_runs

In [ ]:
resolved_shard_roots = split_paths(SHARD_ROOTS)
auto_resolved_shard_group = {"skipped": True, "reason": "manual_shard_roots_provided"}
if not resolved_shard_roots and AUTO_RESOLVE_SHARD_ROOTS:
    resolved_shard_roots, auto_resolved_shard_group = resolve_latest_complete_shard_roots(
        SHARD_RUNS_ROOT,
        required_short_commit=REQUIRED_SHARD_SHORT_COMMIT,
        required_shard_count=REQUIRED_SHARD_COUNT,
    )
if not resolved_shard_roots:
    raise ValueError("SHARD_ROOTS 为空且自动搜索未启用或未找到完整 shard 组。")
print({
    "resolved_shard_roots": resolved_shard_roots,
    "auto_resolved_shard_group": auto_resolved_shard_group,
})
family_id = f"real_video_vae_latent_probe_aggregated_{short_commit}"
local_family_root = Path(LOCAL_FAMILY_ROOT) / family_id
command = [
    sys.executable,
    "scripts/package_results/merge_real_video_vae_latent_shards.py",
    "--merged-run-root", MERGED_RUN_ROOT,
    "--family-root", str(local_family_root),
    "--protocol-config", PROTOCOL_CONFIG,
    "--mechanism-config", MECHANISM_CONFIG,
    "--runtime-profile", RUNTIME_PROFILE,
    "--short-commit", short_commit,
]
if EXCLUDE_LARGE_INTERMEDIATE_LATENTS:
    command.append("--exclude-large-intermediate-latents")
for shard_root in resolved_shard_roots:
    command.extend(["--shard-root", shard_root])
completed = run_command(command, cwd=REPO_DIR)
aggregation_summary = parse_json_process_stdout(completed.stdout)
print(aggregation_summary)


## 3. 落盘到 Drive shard_aggregated

In [ ]:
drive_family_root = Path(DRIVE_ROOT) / "TSTW" / "results" / "real_video_vae_latent_probe" / "shard_aggregated" / local_family_root.name
if drive_family_root.exists():
    if not OVERWRITE_DRIVE_RESULT:
        raise FileExistsError(drive_family_root)
    shutil.rmtree(drive_family_root)
drive_family_root.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_family_root, drive_family_root)
print({"drive_family_root": str(drive_family_root)})

# 使用阶段三的真实输入检查器验证聚合结果是否可被 baseline comparison 直接消费。
downstream_check_path = drive_family_root / "stage_two_package_for_baseline_check.json"
downstream_completed = run_command([
    sys.executable,
    "scripts/prepare_baselines/check_real_video_vae_package_for_baseline.py",
    "--package-root", str(drive_family_root),
    "--summary-path", str(downstream_check_path),
], cwd=REPO_DIR)
downstream_stage_two_check = parse_json_process_stdout(downstream_completed.stdout)
print({"downstream_stage_two_check": downstream_stage_two_check})


## 4. 最终完成性判断

In [ ]:
required_paths = {
    "family_summary": drive_family_root / "family_summary.json",
    "family_checks": drive_family_root / "family_checks.json",
    "family_manifest": drive_family_root / "family_manifest.json",
    "packages": drive_family_root / "packages",
}
completion_summary = {
    "result_flow": "shard_aggregate",
    "drive_family_root": str(drive_family_root),
    "required_paths": {name: path.exists() for name, path in required_paths.items()},
    "input_shard_roots": resolved_shard_roots,
    "auto_resolved_shard_group": auto_resolved_shard_group,
    "ready_for_baseline_comparison_gate": bool(downstream_stage_two_check.get("package_ready_for_baseline_comparison")),
    "downstream_stage_two_check_path": str(downstream_check_path),
    "downstream_stage_two_check": downstream_stage_two_check,
}
completion_summary["status"] = all(completion_summary["required_paths"].values()) and completion_summary["ready_for_baseline_comparison_gate"]
print(json.dumps(completion_summary, ensure_ascii=False, indent=2))
if not completion_summary["status"]:
    raise RuntimeError(completion_summary)
